In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

USER_DATA = {
    "user_a":{
        "a1":{"name":"张三","goods":"电脑"},
        "a2":{"name":"李四","goods":"电视"}
    },
    "user_b":{
        "b1":{"name":"王五","goods":"冰箱"},
        "b2":{"name":"赵六","goods":"空调"}
    }
}

from dataclasses import dataclass

@dataclass(frozen=True)
class UserContext:
    user_id: str
    role_id: str

from langchain.tools import tool,ToolRuntime

@tool
def get_current_user_data(runtime: ToolRuntime[UserContext]):
    """查询当前登录用户的资源"""
    ctx = runtime.context
    data = USER_DATA.get(ctx.role_id,{}).get(ctx.user_id)
    return f"姓名:{data['name']},商品:{data['goods']}"

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="qwen3.5-plus",
    model_provider="openai",
    api_key = os.getenv("DASHSCOPE_API_KEY"),
    base_url = os.getenv("DASHSCOPE_BASE_URL")
)

agent = create_agent(
    model,
    tools=[get_current_user_data],
    context_schema=UserContext,
    system_prompt="你是购物助手，调用工具来查询用户信息"
)

from langchain_core.messages import HumanMessage

response = agent.invoke(
    {"messages":[HumanMessage("查询一下我购买的商品")]},
    context=UserContext(user_id="b1",role_id="user_b")
)

for message in response['messages']:
    message.pretty_print()


================================ Human Message =================================

查询一下我购买的商品
================================== Ai Message ==================================
Tool Calls:
  get_current_user_data (call_a097515e49c444af80c0a3c8)
 Call ID: call_a097515e49c444af80c0a3c8
  Args:
================================= Tool Message =================================
Name: get_current_user_data

姓名:王五,商品:冰箱
================================== Ai Message ==================================

您好，王五！根据查询结果，您购买的商品是冰箱。
